# प्रकरण 7: चॅट अनुप्रयोग तयार करणे
## Azure OpenAI API जलद प्रारंभ


## आढावा
हे नोटबुक [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) मधून रुपांतरित केले आहे ज्यामध्ये अशा नोटबुक्सचा समावेश आहे जे [OpenAI](notebook-openai.ipynb) सेवा देखील प्रवेश करतात.

Python OpenAI API मध्ये Azure OpenAI सहही काही सुधारणा करून कार्य करते. येथे फरक जाणून घ्या: [Python सह OpenAI आणि Azure OpenAI एंडपॉइंट्समध्ये कसे स्विच करावे](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

अधिक तत्पर प्रारंभ उदाहरणांसाठी कृपया अधिकृत [Azure OpenAI Quickstart Documentation](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) पहा


## अनुक्रमणिका  

[आढावा](#overview)  
[Azure OpenAI सेवा सुरू करणे](#getting-started-with-azure-openai-service)  
[आपला पहिले प्रॉम्प्ट तयार करा](#build-your-first-prompt)  

[वापर केसेस](#use-cases)    
[1. मजकूर संक्षेप करा](#summarize-text)  
[2. मजकूर वर्गीकरण करा](#classify-text)  
[3. नवीन उत्पाद नाव तयार करा](#generate-new-product-names)  
[4. वर्गीकाराला सूक्ष्मसुधार करा](#fine-tune-a-classifier)  
[5. एम्बेडिंग्ज](#embeddings)

[संदर्भ](#references)


### Azure OpenAI सेवा सुरू करण्यासाठी मार्गदर्शक

नवीन ग्राहकांनी Azure OpenAI सेवेचा वापर करण्यासाठी [अॅक्सेससाठी अर्ज करणे](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) आवश्यक आहे.  
मंजुरी नंतर, ग्राहक Azure पोर्टलमध्ये लॉगिन करू शकतात, Azure OpenAI सेवा संसाधन तयार करू शकतात, आणि स्टुडिओमार्फत मॉडेल्ससह प्रयोग करु शकतात.  

[जलद सुरू करण्यासाठी उत्कृष्ट स्रोत](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### तुमचा पहिला प्रॉम्प्ट तयार करा  
हा लहान व्यायाम "सारांश" या साध्या कामासाठी OpenAI मॉडेलला प्रॉम्प्ट सबमिट करण्यासाठी मूलभूत ओळख देईल.


**पावले**:  
1. तुमच्या पायथन वातावरणात OpenAI लायब्ररी इंस्टॉल करा  
2. सामान्य सहाय्यक लायब्ररी लोड करा व तुम्ही तयार केलेल्या OpenAI सेवेसाठी तुमची सामान्य OpenAI सुरक्षा प्रमाणपत्रे सेट करा  
3. तुमच्या कामासाठी एक मॉडेल निवडा  
4. मॉडेलसाठी एक सोपा प्रॉम्प्ट तयार करा  
5. तुमचा विनंती मॉडेल API कडे सबमिट करा!


### 1. OpenAI स्थापित करा


  > [!NOTE] जर हा नोटबुक Codespaces वर किंवा Devcontainer मध्ये चालवला गेला तर हा टप्पा आवश्यक नाही


In [ ]:
%pip install openai python-dotenv

### 2. सहाय्यक लायब्ररी आयात करा आणि प्रमाणपत्रे तयार करा


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. योग्य मॉडेल शोधणे  
GPT-4o आणि GPT-4o मिनी सारखी मॉडेल्स नैसर्गिक भाषा समजू शकतात आणि तयार करू शकतात. मायक्रोसॉफ्ट फाउंड्रीमधील Azure OpenAI वेगवेगळ्या कार्यांसाठी वेगवेगळ्या क्षमतांसह, वेगवेगळ्या सामर्थ्य आणि गतीसह विविध मॉडेल क्षमता ऑफर करतो. 

[मायक्रोसॉफ्ट फाउंड्रीमधील Azure OpenAI मॉडेल्स](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. प्रॉम्प्ट डिझाईन  

"मोठ्या भाषा मॉडेल्सचा जादू म्हणजे मोठ्या प्रमाणावर मजकूरावर ह्या भाकीत त्रुटी कमी करण्यासाठी प्रशिक्षित होऊन, मॉडेल अशा संकल्पना शिकतात ज्या या भाकीतसाठी उपयुक्त आहेत. उदाहरणार्थ, ते अशा संकल्पना शिकतात"(1):

* शब्दशुद्धी कशी करावी
* व्याकरण कसे कार्य करते
* पर्यायी शब्दसंग्रह कसा करावा
* प्रश्नांची उत्तरे कशी द्यावी
* संभाषण कसे सुरू ठेवावे
* अनेक भाषांमध्ये लेखन कसे करावे
* कोडिंग कसे करावे
* इत्यादी.

#### मोठ्या भाषा मॉडेलवर कसे नियंत्रण ठेवावे  
"मोठ्या भाषा मॉडेलसाठी सर्वात प्रभावी इनपुट म्हणजे मजकूर प्रॉम्प्ट(1).

मोठ्या भाषा मॉडेल्सना आउटपुट तयार करण्यासाठी काही मार्गांनी प्रॉम्प्ट देता येऊ शकतो:

- सूचना: मॉडेलला तुम्हाला काय हवे ते सांगा
- पूर्णता: तुम्हाला हवे असलेल्या प्रारंभाची पूर्णता करण्यासाठी मॉडेलला प्रवृत्त करा
- प्रदर्शन: मॉडेलला काय हवे आहे ते दाखवा, खालीलपैकी कोणत्याही एका प्रकारे:
- प्रॉम्प्टमध्ये काही उदाहरणे
- फाइन-ट्यूनिंग प्रशिक्षण डेटासेटमध्ये शेकडो किंवा हजारो उदाहरणे



#### प्रॉम्प्ट तयार करताना तीन मूलभूत मार्गदर्शक तत्त्वे आहेत:

**दाखवा आणि सांगा**. तुम्हाला काय हवे आहे ते सूचना, उदाहरणे, किंवा दोन्हींच्या संयोगाने स्पष्ट करा. जर तुम्हाला मॉडेलला वस्तूंची यादी अक्षरक्रमाने क्रमवारी लावण्यास सांगायचे असेल किंवा परिच्छेदाचे भावना अनुसार वर्गीकरण करायचे असेल, तर त्याला ते दाखवा.

**चांगला डेटा पुरवा**. जर तुम्ही वर्गीकरणकर्ता तयार करण्याचा प्रयत्न करत असाल किंवा मॉडेलला एखाद्या नमुन्याचे पालन करण्यास सांगत असाल, तर पुरेसे उदाहरणे असतील याची खात्री करा. तुमची उदाहरणे नाळणे लक्षात ठेवा — मॉडेल सहसा मूलभूत शब्दशुद्धी चुका ओळखून उत्तर देते, पण कधी-कधी ते चुकीचे समजून उत्तरावर परिणाम करू शकते.

**तुमची सेटिंग तपासा.** तापमान आणि टॉप_p सेटिंग्ज मॉडेलच्या प्रतिसादनिर्मितीतील निश्चितता नियंत्रित करतात. जर तुम्ही असे उत्तर विचारत असाल ज्यासाठी फक्त एकच बरोबर उत्तर असेल, तर तुम्ही हे कमी सेट कराल. जर तुम्हाला अधिक विविध प्रतिसाद हवे असतील, तर तुम्ही ते जास्त सेट कराल. लोक याबाबतीत केलेली सर्वांत मोठी चूक म्हणजे हे सेटिंग्ज "चतुराई" किंवा "सर्जनशीलता" नियंत्रित करतात असा समज राखणे.


स्रोत: https://learn.microsoft.com/azure/ai-foundry/openai/overview


चित्र आपला पहिला मजकूर प्रॉम्प्ट तयार करत आहे!


### 5. सबमिट करा!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### सारखे कॉल पुन्हा करा, परिणाम कसे तुलना करतात? 


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## मजकूर सारांशित करा  
#### आव्हान  
मजकूराच्या शेवटी 'tl;dr:' जोडून मजकूराचा सारांश तयार करा. लक्षात घ्या की हा मॉडेल कशाप्रकारे कोणतीही अतिरिक्त सूचना न देता अनेक कामे कशी पार पाडते हे समजून घेतो. तुम्ही tl;dr पेक्षा अधिक वर्णनात्मक प्रॉम्प्ट वापरून मॉडेलच्या वर्तनात बदल करू शकता आणि तुम्हाला मिळणाऱ्या सारांश रूपांतरणाचा सानुकूल करू शकता(3).  

अलीकडील कामाने मोठ्या लेखनसंग्रहावर प्री-ट्रेनिंग करून आणि नंतर विशिष्ट कामावर फाइन-ट्यूनिंग करून अनेक NLP कामांमध्ये आणि बेंचमार्कमध्ये लक्षणीय वाढ दाखवली आहे. जरी हे सहसा आर्किटेक्चरमध्ये काम-रहित असले तरी, या पद्धतीस हजारो किंवा दहा हजारो उदाहरणांच्या काम-विशिष्ट फाइन-ट्यूनिंग डेटासेटची आवश्यकता असते. त्याउलट, माणसे साधारणपणे काही उदाहरणे किंवा सोप्या सूचनांवरून नवीन भाषा काम करू शकतात - जे सध्याच्या NLP प्रणाली अद्याप मोठ्या प्रमाणावर करू शकत नाहीत. येथे आम्ही दाखवतो की भाषा मॉडेलचे विस्तारीकरण काम-रहित, कमी उदाहरणांवर कामगिरी फार सुधारते, कधीकधी अगदी पूर्वीच्या कार्यक्षम फाइन-ट्यूनिंग पद्धतींच्या स्पर्धात्मक पातळीवर पोहोचते.  



Tl;dr


# विविध वापर प्रकरणांसाठी सराव  
1. मजकूराचा सारांश तयार करा  
2. मजकूर वर्गीकरण  
3. नवीन उत्पादन नावे तयार करा  
4. एम्बेडिंग्ज  
5. वर्गीकारकाचं फाईन ट्यूनिंग करा  


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## मजकूर वर्गीकरण करा  
#### आव्हान  
वस्तूना वर्गीकृत करा ज्यासाठी श्रेण्या अनुमानाच्या वेळी दिल्या जातात. पुढील उदाहरणात, आम्ही श्रेण्या आणि वर्गीकरणासाठी मजकूर दोन्हीच प्लेग्राउंड_रेफरन्समध्ये प्रदान केले आहेत. 

ग्राहक चौकशी: नमस्कार, माझ्या लॅपटॉपच्या कीबोर्डवरील एक की नुकतीच तुटली आहे आणि मला तिचा बदलावा लागेल:

वर्गीकृत श्रेणी:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## नवीन उत्पाद नावे तयार करा
#### आव्हान
उदाहरण शब्दांमधून उत्पादनांची नावे तयार करा. येथे आम्ही उत्पादनासाठी नावे निर्माण करत असलेल्या माहितीचा समावेश प्रॉम्प्टमध्ये करतो. आम्ही इच्छित नमुना दाखविण्यासाठी एक समान उदाहरण देखील पुरवतो. आम्ही यादृच्छिकता आणि अधिक नवोन्मेषी प्रतिसाद वाढविण्यासाठी तापमानाचे मूल्य जास्त ठेवले आहे.

उत्पादनाचे वर्णनः एक घरगुती दुधशेक मेकर
बीज शब्दः जलद, आरोग्यदायी, कॉम्पॅक्ट.
उत्पादन नावे: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पादनाचे वर्णनः एक जोडी बूट जी कोणत्याही पायाच्या आकाराला बसू शकते.
बीज शब्दः अनुकूलनीय, योग्य, सर्वसामान्य-फिट.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## एम्बेडिंग्ज
हा विभाग कसा एम्बेडिंग्ज मिळवायचा, आणि शब्द, वाक्ये, आणि दस्तऐवज यांच्यातील साम्य शोधायचे याचे उदाहरण दर्शवेल. खालील नोटबुक चालविण्यासाठी तुम्हाला `text-embedding-ada-002` या बेस मॉडेलचा वापर करणारे मॉडेल डिप्लॉय करावे लागेल आणि त्याचा डिप्लॉयमेंट नाव .env फाइलमध्ये `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT` या वेरिएबलमध्ये सेट करावा लागेल.


### मॉडेल वर्गीकरण - एम्बेडिंग मॉडेल निवडणे

**मॉडेल वर्गीकरण**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (एम्बेडिंग कुटुंब)  
{capability} --> ada             (इतर सर्व एम्बेडिंग मॉडेल्स 2024 मध्ये निवृत्त होतील)  
{input-type} --> n/a             (फक्त शोध मॉडेलसाठी निर्दिष्ट केलेले)  
{identifier} --> 002             (आवृत्ती 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] पुढील पायरी आवश्यक नाही जर ही नोटबुक Codespaces वर किंवा Devcontainer मध्ये चालवली जात असेल


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## cnn daily news डेटासेटमधील लेखाची तुलना
source: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# संदर्भ  
- [Microsoft Foundry - Azure OpenAI Models](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# अधिक मदतीसाठी  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# योगदानकर्ते
* लुईस ली  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
